In [2]:
import json
from pathlib import Path

import os

import pandas as pd

from data.croissant_data_loader import CroissantDataLoader

CROISSANT_FOLDER_PATH = 'datasets/croissant'
SEED = 42

# Get dataset information for each dataset
for filename in os.listdir(Path(CROISSANT_FOLDER_PATH)):
    if not filename.endswith('.json'):
        continue

    croissant_file_path = Path(CROISSANT_FOLDER_PATH, filename)
    dataloader = CroissantDataLoader(croissant_file_path, seed=SEED)
    shot_id_col = dataloader.shot_column
    target_col = dataloader.target_column
    dataset_name = dataloader.dataset_name
    dataset_info = {
        'dataset_name': dataset_name,
    }

    train_df = dataloader.get_train_df()
    val_df = dataloader.get_val_df()
    test_df = dataloader.get_test_df()
    train_and_test_df = pd.concat([train_df, val_df])

    def get_info_for_df(df: pd.DataFrame) -> dict:
        # Get shot count and proportion of logical errors
        n_shots = df[shot_id_col].nunique()
        assert n_shots == df.shape[0] # shot_id should be unique
        logical_error_fraction = df[target_col].sum() / n_shots
        num_round_col = dataloader.num_rounds_column
        unique_num_rounds = sorted(df[num_round_col].unique().tolist())
        return {
            'n_shots': n_shots,
            'logical_error_fraction': logical_error_fraction,
            'no_logical_error_fraction': 1 - logical_error_fraction,
            'unique_num_rounds': unique_num_rounds,
        }

    dataset_info['info_per_split'] = { split: get_info_for_df(df) for split, df in zip(['train', 'val', 'test', 'train+val'], [train_df, val_df, test_df, train_and_test_df]) }

    feature_group_mapping = dataloader.get_feature_group_mapping()
    feature_counts = { k: len(v) for k, v in feature_group_mapping.items() }
    feature_counts['total'] = sum(feature_counts.values())
    dataset_info['feature_info'] = {
        'features': list(feature_group_mapping.keys()),
        'feature_counts': feature_counts
    }

    # Find path containing actual data
    data_folder = Path(dataloader.croissant_metadata['distribution'][0]['contentUrl']).parent

    # Save dataset information
    save_path = data_folder / 'dataset_info.json'
    with open(save_path, 'w') as f:
        json.dump(dataset_info, f, indent=4)
        print('Wrote dataset info to {}'.format(save_path))


Loading training data from datasets\dicarlo_rep_d3\train.parquet
Loading test data from datasets\dicarlo_rep_d3\test.parquet
Loading raw data from datasets\dicarlo_rep_d3\raw.parquet
Wrote dataset info to datasets\dicarlo_rep_d3\dataset_info.json
Loading training data from datasets\google_surface_code\d3\train.parquet
Loading test data from datasets\google_surface_code\d3\test.parquet
Loading raw data from datasets\google_surface_code\d3\raw.parquet
Wrote dataset info to datasets\google_surface_code\d3\dataset_info.json
Loading training data from datasets\google_surface_code\d5\train.parquet
Loading test data from datasets\google_surface_code\d5\test.parquet
Loading raw data from datasets\google_surface_code\d5\raw.parquet
Wrote dataset info to datasets\google_surface_code\d5\dataset_info.json
Loading training data from datasets\google_rep_code\d25\train.parquet
Loading test data from datasets\google_rep_code\d25\test.parquet
Loading raw data from datasets\google_rep_code\d25\raw.parqu